# Run the staged spectral-calibration workflow

Configure the three training stages, then execute preprocessing, staged training, and evaluation. `START_FROM = 2` reuses prepared datasets. `SMOKE = True` selects the original 20-epoch configuration for each stage; it still runs actual training.

**Runtime dependency:** the supplied archive does not contain `spice_moe.py`. Recover the matching version before execution. See [reproducibility notes](../docs/REPRODUCIBILITY.md).

Run configuration and executed notebook copies are saved inside a separate experiment directory. Original training defaults are preserved.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path(os.environ["MOE_PROJECT_ROOT"]) if "MOE_PROJECT_ROOT" in os.environ else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "scripts" / "project_paths.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or its notebooks directory.")
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "src"))
from project_paths import configure_run
DATA_ROOT, RUN_ROOT, PREPARED_DIR, CHECKPOINT_DIR = configure_run(new=True)


In [ ]:
# === 00 / one-click runner — ALL SETTINGS LIVE HERE ======================
import os, re, sys, time, shutil, datetime, json
import nbformat
from nbclient import NotebookClient
from IPython.display import display, HTML, clear_output
sys.path.insert(0, os.getcwd())
import spice_moe as S

HERE = str(ROOT / "notebooks")
NOTEBOOKS = ["01_data_preprocessing.ipynb",
             "02_stage1_naca.ipynb",
             "03_stage2_add_k.ipynb",
             "04_stage3_add_mg.ipynb",
             "05_testing_evaluation.ipynb"]

# ---------------------------------------------------------------- which to run
START_FROM = 1      # 1 = rebuild ML dataset;  2 = reuse it and start training
STOP_AFTER = 5
BACKUP     = True   # keep the executed notebooks in _runs/<timestamp>/
TIMEOUT    = None   # seconds per cell; None = no limit

# ------------------------------------------------------- training configuration
# Edited here, written to run_config.json, read by 02/03/04 via S.stage_settings().
# Running 02/03/04 on their own still works: they fall back to the same defaults.
SMOKE = False       # True -> every stage runs SMOKE_CFG instead (quick sanity check)

TRAIN_CFG = {
    "smoke":     SMOKE,
    "stage1":    {"epochs": 1200, "patience": 250, "batch": 512},   # trains everything
    "stage2":    {"epochs":  800, "patience": 200, "batch": 512},   # only the K expert
    "stage3":    {"epochs":  800, "patience": 200, "batch": 512},   # only the Mg expert
    "smoke_cfg": {"epochs":   20, "patience":  20, "batch": 256},

    # Optional loss-weight overrides.  Leave empty to use spice_moe.W.
    # If anchor_rel stops falling and dominates the gradient, try 10.0 here.
    "weights": {},          # e.g. {"anchor_rel": 10.0}
}
print("wrote", S.save_run_config(TRAIN_CFG, str(RUN_ROOT)))

# NOTE ON LEARNING RATE.  02/03/04 set the cosine decay to
#   total_steps = EPOCHS * STEPS_PER_EPOCH
# so the schedule finishes exactly at EPOCHS.  Setting EPOCHS far higher than the
# run will actually reach (the old 3000) leaves the learning rate near its initial
# value for the whole run, and the systematic-bias term never fine-converges.
# Keep EPOCHS close to what you really intend to run.

# ------------------------------------------------------------- time estimate
SEC_PER_EPOCH = {"stage1": 36.6, "stage2": 50.0, "stage3": 50.0}   # measured / estimated
tot = 0.0
print(f"\n{'stage':8}{'epochs':>8}{'patience':>10}{'batch':>7}{'s/epoch':>9}{'max time':>11}")
for st in ("stage1", "stage2", "stage3"):
    c = TRAIN_CFG["smoke_cfg"] if SMOKE else TRAIN_CFG[st]
    h = c["epochs"] * SEC_PER_EPOCH[st] / 3600; tot += h
    print(f"{st:8}{c['epochs']:>8}{c['patience']:>10}{c['batch']:>7}"
          f"{SEC_PER_EPOCH[st]:>9.1f}{h:>10.1f}h")
print(f"{'TOTAL':8}{'':>34}{tot:>10.1f}h   (early stopping usually finishes sooner)")


In [ ]:
# === 00 / progress display ===============================================
EPOCH_RE = re.compile(r"\[([\w\-]+)\]\s+Epoch\s+(\d+)\s*/\s*(\d+).*?time=\s*([\d.]+)s")

class Progress:
    """Parses the training log and renders a bar.  Also keeps the last few lines so
    the loss values stay visible while the bar updates."""
    def __init__(self, total_nb):
        self.total_nb = total_nb; self.nb_i = 0; self.nb_name = ""
        self.label = ""; self.ep = 0; self.n_ep = 0; self.sec = 0.0
        self.t0 = time.time(); self.tail = []

    def feed(self, text):
        for line in text.splitlines():
            if line.strip():
                self.tail.append(line.rstrip())
            m = EPOCH_RE.search(line)
            if m:
                self.label, self.ep, self.n_ep, self.sec = \
                    m.group(1), int(m.group(2)), int(m.group(3)), float(m.group(4))
        self.tail = self.tail[-12:]
        self.render()

    def render(self):
        frac = self.ep / self.n_ep if self.n_ep else 0.0
        eta = (self.n_ep - self.ep) * self.sec if self.n_ep and self.sec else 0
        bar_nb = int(60 * (self.nb_i - 1 + frac) / self.total_nb)
        bar_ep = int(60 * frac)
        el = int(time.time() - self.t0)
        clear_output(wait=True)
        display(HTML(f"""
        <div style="font-family:system-ui,-apple-system,'Segoe UI',sans-serif;font-size:13px">
          <b>Notebook {self.nb_i}/{self.total_nb}</b> &nbsp; {self.nb_name}
          <div style="background:#eceff4;border-radius:6px;height:14px;margin:5px 0;width:520px">
            <div style="background:#2b5c9c;height:14px;border-radius:6px;
                        width:{bar_nb/60*520:.0f}px"></div></div>
          <b>{self.label or 'running'}</b> &nbsp; epoch {self.ep}/{self.n_ep}
          &nbsp;<span style="color:#666">{self.sec:.1f}s/epoch &middot;
          elapsed {el//60}m{el%60:02d}s &middot; eta {int(eta)//60}m</span>
          <div style="background:#eceff4;border-radius:6px;height:10px;margin:5px 0;width:520px">
            <div style="background:#238b45;height:10px;border-radius:6px;
                        width:{bar_ep/60*520:.0f}px"></div></div>
          <pre style="background:#0f172a;color:#cbd5e1;padding:8px;border-radius:6px;
               max-height:190px;overflow:auto;font-size:11px;width:660px">{
               chr(10).join(self.tail)}</pre>
        </div>"""))

class Runner(NotebookClient):
    """NotebookClient that forwards every stream output to the progress display."""
    prog = None
    def output(self, outs, msg, display_id, cell_index):
        if self.prog is not None and msg.get("msg_type") == "stream":
            self.prog.feed(msg["content"].get("text", ""))
        return super().output(outs, msg, display_id, cell_index)


In [ ]:
# === 00 / run ============================================================
todo = NOTEBOOKS[START_FROM - 1:STOP_AFTER]
prog = Progress(len(todo))
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = os.path.join(RUN_ROOT, "executed_notebooks", stamp)
if BACKUP: os.makedirs(run_dir, exist_ok=True)

failed = None
for i, nb_name in enumerate(todo, start=1):
    prog.nb_i, prog.nb_name = i, nb_name
    prog.ep = prog.n_ep = 0
    prog.tail.append(f"===== {nb_name} =====")
    prog.render()
    nb = nbformat.read(os.path.join(HERE, nb_name), as_version=4)
    client = Runner(nb, timeout=TIMEOUT, kernel_name="python3",
                    resources={"metadata": {"path": str(RUN_ROOT)}},
                    allow_errors=False)
    client.prog = prog
    try:
        client.execute()
    except Exception as e:
        failed = (nb_name, e)
        prog.tail.append(f"!! FAILED in {nb_name}: {type(e).__name__}: {e}")
        prog.render()
        break
    finally:
        if BACKUP:
            nbformat.write(nb, os.path.join(run_dir, nb_name))

if failed:
    print(f"\nStopped at {failed[0]}:\n{failed[1]}")
else:
    el = int(time.time() - prog.t0)
    print(f"\nAll {len(todo)} notebooks finished in {el//3600}h{(el%3600)//60:02d}m.")
    if BACKUP: print("executed copies saved to", run_dir)
